# Sign-to-Text Training (Colab)

Runtime -> Change runtime type -> **GPU** (T4 is fine to start).

This notebook: clones your GitHub repo, installs dependencies, mounts
Google Drive (so the dataset + checkpoints survive when the Colab VM
recycles), sanity-checks the data, then trains.


## 1. Clone your repo
Replace the URL with your own GitHub repo once you've pushed it.

In [ ]:
!git clone https://github.com/<your-username>/sign-to-text-slt.git
%cd sign-to-text-slt


## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 3. Mount Google Drive

Colab's local disk is wiped every time the runtime disconnects. Store the
(large, licensed) PHOENIX-2014-T dataset and your checkpoints on Drive so
you don't have to re-download/re-train from zero each session.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this to wherever you've put (or will put) the extracted dataset on Drive
DATA_ON_DRIVE = "/content/drive/MyDrive/phoenix2014T"
CKPT_ON_DRIVE = "/content/drive/MyDrive/slt_checkpoints"

import os
os.makedirs(CKPT_ON_DRIVE, exist_ok=True)

# symlink so the repo's default relative paths (data/, checkpoints/) just work
!rm -rf data checkpoints
!ln -s "$DATA_ON_DRIVE" data_phoenix2014T_link
!mkdir -p data
!ln -s "$DATA_ON_DRIVE" data/phoenix2014T
!ln -s "$CKPT_ON_DRIVE" checkpoints


## 4. First-time only: get the dataset onto Drive

RWTH-PHOENIX-2014-T requires requesting access (it's not a public direct
download) -- see the README. Once you have the archive, the easiest path
is uploading it straight into `MyDrive/phoenix2014T` via the Drive web UI
or `gdown`/`wget` if you have a direct link, then extracting here:


In [ ]:
# Example if you've uploaded a .zip/.tar to Drive already:
# !unzip -q /content/drive/MyDrive/phoenix2014T.zip -d /content/drive/MyDrive/phoenix2014T


## 5. Sanity-check the data before training

In [ ]:
!python -m src.inspect_data --config configs/config.yaml


## 6. Train
Checkpoints save every epoch to `checkpoints/`, which is symlinked to Drive -- safe against Colab disconnects.

In [ ]:
!python -m src.train --config configs/config.yaml


## 7. Evaluate

In [ ]:
!python -m src.evaluate --config configs/config.yaml \
    --checkpoint checkpoints/epoch30.pt --split test
